# Building Demolition Risk Assessment: Comparative Study
## Approach 1 (Deep Neural Network / GA-MLP) vs. Approach 2 (Multi-Class SVM)
**Project:** Building Demolition Risk Evaluation (Dr. Saghafi)  
**Target:** 22-class risk assessment based on 9 Likert-scale criteria ($x_1$ to $x_9$).  

### Overview
- **Approach 1 (Neural Network):** Multilayer Perceptron (MLP) optimized via Genetic Algorithm (Neurons: 57, 115; Dropout: 0.0914; Learning Rate: 0.00579).
- **Approach 2 (Support Vector Machine):** Maximum-margin non-linear classifier with RBF Kernel and One-vs-One multi-class decomposition (231 pairwise hyperplanes).

In [1]:
# 1. Environment & Library Imports
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

print("Libraries successfully imported!")

Libraries successfully imported!


In [2]:
# 2. Data Loading & Preprocessing
data_path = 'Final_Dataset.xlsx'
df = pd.read_excel(data_path)

feature_cols = df.columns[:9]
target_col = df.columns[-1]

X = df[feature_cols].values.astype(np.float64)
y_raw = df[target_col].values.astype(int)

# Ensure 0-indexed labels (0 to 21)
y = y_raw - 1 if y_raw.min() == 1 else y_raw

print(f"Dataset Shape: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Number of Classes: {len(np.unique(y))} (Classes 0 to 21)")

# Stratified 70/30 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler applied successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'Final_Dataset.xlsx'

In [ ]:
# 3. Approach 2: Support Vector Machine (SVM) Optimization & Training
param_grid = [
    {'kernel': ['rbf'], 'C': [0.1, 1, 10, 50, 100], 'gamma': ['scale', 'auto', 0.05, 0.1]},
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
    {'kernel': ['poly'], 'C': [1, 10], 'degree': [2, 3]}
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_svm = GridSearchCV(SVC(decision_function_shape='ovo', random_state=42), param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)

print("Running GridSearchCV for SVM...")
start_t = time.time()
grid_svm.fit(X_train_scaled, y_train)
svm_tune_time = time.time() - start_t

best_svm = grid_svm.best_estimator_
print(f"Best SVM Parameters: {grid_svm.best_params_}")
print(f"Best 5-Fold CV Accuracy: {grid_svm.best_score_:.4f} (Time: {svm_tune_time:.2f}s)")

# Evaluate SVM on Test Set
start_inf = time.time()
y_pred_svm = best_svm.predict(X_test_scaled)
svm_latency = (time.time() - start_inf) / len(X_test_scaled) * 1000

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='macro')
print(f"SVM Test Accuracy: {acc_svm:.4f} | Macro F1: {f1_svm:.4f} | Latency: {svm_latency:.3f} ms/sample")

In [ ]:
# 4. Approach 1: GA-Optimized Neural Network (MLP) Training
# Architecture corresponds to the best GA parameters: (57, 115) neurons, Adam lr=0.00579
mlp = MLPClassifier(
    hidden_layer_sizes=(57, 115),
    activation='relu',
    solver='adam',
    learning_rate_init=0.00579,
    max_iter=400,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=15
)

print("Training GA-Optimized Neural Network...")
start_nn = time.time()
mlp.fit(X_train_scaled, y_train)
train_time_nn = time.time() - start_nn

start_inf_nn = time.time()
y_pred_nn = mlp.predict(X_test_scaled)
nn_latency = (time.time() - start_inf_nn) / len(X_test_scaled) * 1000

acc_nn = accuracy_score(y_test, y_pred_nn)
f1_nn = f1_score(y_test, y_pred_nn, average='macro')
print(f"Neural Network Test Accuracy: {acc_nn:.4f} | Macro F1: {f1_nn:.4f} | Latency: {nn_latency:.3f} ms/sample")

In [ ]:
# 5. Side-by-Side Confusion Matrix Visualization
cm_nn = confusion_matrix(y_test, y_pred_nn)
cm_svm = confusion_matrix(y_test, y_pred_svm)
classes = [f"C{i}" for i in range(22)]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm_nn, annot=False, cmap='Blues', ax=axes[0], xticklabels=classes, yticklabels=classes)
axes[0].set_title("Neural Network (GA-Optimized MLP)\nConfusion Matrix", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("True Class")

sns.heatmap(cm_svm, annot=False, cmap='Greens', ax=axes[1], xticklabels=classes, yticklabels=classes)
axes[1].set_title("Support Vector Machine (Optimized RBF)\nConfusion Matrix", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("True Class")

plt.tight_layout()
plt.show()

In [ ]:
# 6. Class-by-Class F1 Score Comparison (22 Classes)
rep_nn = classification_report(y_test, y_pred_nn, output_dict=True, zero_division=0)
rep_svm = classification_report(y_test, y_pred_svm, output_dict=True, zero_division=0)

f1_list_nn = [rep_nn[str(i)]['f1-score'] if str(i) in rep_nn else 0.0 for i in range(22)]
f1_list_svm = [rep_svm[str(i)]['f1-score'] if str(i) in rep_svm else 0.0 for i in range(22)]

x = np.arange(22)
width = 0.38

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - width/2, f1_list_nn, width, label='Neural Network (MLP)', color='#2980B9', alpha=0.9)
ax.bar(x + width/2, f1_list_svm, width, label='Support Vector Machine (SVM)', color='#27AE60', alpha=0.9)

ax.set_xlabel('Class (0 to 21)', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score Comparison: Neural Network vs. SVM', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"C{i}" for i in range(22)], rotation=45)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Overall Summary Comparison Table
metrics_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1-Score', 'Weighted F1-Score'],
    'Neural Network (MLP)': [
        accuracy_score(y_test, y_pred_nn),
        precision_score(y_test, y_pred_nn, average='macro', zero_division=0),
        recall_score(y_test, y_pred_nn, average='macro', zero_division=0),
        f1_score(y_test, y_pred_nn, average='macro', zero_division=0),
        f1_score(y_test, y_pred_nn, average='weighted', zero_division=0)
    ],
    'Support Vector Machine (SVM)': [
        accuracy_score(y_test, y_pred_svm),
        precision_score(y_test, y_pred_svm, average='macro', zero_division=0),
        recall_score(y_test, y_pred_svm, average='macro', zero_division=0),
        f1_score(y_test, y_pred_svm, average='macro', zero_division=0),
        f1_score(y_test, y_pred_svm, average='weighted', zero_division=0)
    ]
})
metrics_table['Difference (SVM - NN)'] = metrics_table['Support Vector Machine (SVM)'] - metrics_table['Neural Network (MLP)']
metrics_table['Winner'] = np.where(metrics_table['Difference (SVM - NN)'] > 0, 'SVM', 'NN')

display(metrics_table)